In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

The model predicts NBA career impact. I created one combined impact score using BPM, VORP, and Win Shares.

In [3]:
df = pd.read_csv("tableau_nba_draft_master.csv")

df["nba_impact_score"] = (
    df["BPM"].fillna(0) * 2
    + df["VORP"].fillna(0) * 0.5
    + df["WS"].fillna(0) * 0.1
)

# Only train on players who already have NBA career results
model_df = df[df[["BPM", "VORP", "WS"]].notna().any(axis=1)].copy()

print(model_df.shape)

(868, 81)


The model uses draft position, college performance, and combine measurements. NBA career stats are not used as inputs because that would leak the answer.

In [4]:
feature_cols = [
    "Pk",
    "DraftYear",

    "college_year_gap",
    "college_g",
    "college_mpg",
    "college_ppg",
    "college_rpg",
    "college_apg",
    "college_bpm",
    "college_obpm",
    "college_dbpm",
    "college_porpag",
    "college_ortg",
    "college_drtg",
    "college_usg",
    "college_ts",
    "college_efg",
    "college_three_pct",

    "combine_weight_lbs",
    "Standing vertical Jump",
    "Maximum Vertical Jump (with steps)",
    "three-quarter court sprint",
    "lane agility drill",
    "Bench Press test",

    "has_college_stats",
    "has_combine_data"
]

X = model_df[feature_cols]
y = model_df["nba_impact_score"]

I used a Random Forest model because it can capture nonlinear patterns and also provides feature importance.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42
)

model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestRegressor(
        n_estimators=300,
        max_depth=6,
        min_samples_leaf=5,
        random_state=42
    ))
])

model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('imputer', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. If a fe

The model is evaluated using MAE and R-squared. MAE shows the average prediction error, while R-squared shows how much variation the model explains.

In [6]:
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Mean Absolute Error:", round(mae, 2))
print("R-squared:", round(r2, 3))

Mean Absolute Error: 7.71
R-squared: 0.213


This shows which draft, college, and combine variables were most useful for predicting NBA career impact.

In [7]:
feature_importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": model.named_steps["model"].feature_importances_
}).sort_values("importance", ascending=False)

feature_importance.head(15)

,feature,importance
0,Pk,0.463615
1,DraftYear,0.131467
22,lane agility drill,0.053245
18,combine_weight_lbs,0.041463
20,Maximum Vertical Jump (with steps),0.033246
21,three-quarter court sprint,0.026727
10,college_dbpm,0.026566
12,college_ortg,0.023646
8,college_bpm,0.023524
19,Standing vertical Jump,0.020695


The model predictions are added back to the dataset so Tableau can show which players overperformed or underperformed compared to the model's expectations.

In [12]:
prediction_data = df.copy()

prediction_data["nba_impact_score"] = (
    prediction_data["BPM"].fillna(0) * 2
    + prediction_data["VORP"].fillna(0) * 0.5
    + prediction_data["WS"].fillna(0) * 0.1
)

prediction_data["predicted_nba_impact"] = model.predict(
    prediction_data[feature_cols]
)

prediction_data["impact_vs_prediction"] = (
    prediction_data["nba_impact_score"]
    - prediction_data["predicted_nba_impact"]
)

prediction_data["model_result"] = np.where(
    prediction_data["impact_vs_prediction"] >= 5,
    "Overperformed Prediction",
    np.where(
        prediction_data["impact_vs_prediction"] <= -5,
        "Underperformed Prediction",
        "Near Prediction"
    )
)

prediction_data.to_csv("tableau_nba_draft_model_predictions.csv", index=False)

prediction_data[
    ["player", "DraftYear", "Pk", "nba_impact_score", 
     "predicted_nba_impact", "impact_vs_prediction", "model_result"]
].sort_values("impact_vs_prediction", ascending=False).head(10)



,player,DraftYear,Pk,nba_impact_score,predicted_nba_impact,impact_vs_prediction,model_result
280,Nikola Jokić,2014,41,76.74,0.319409,76.420591,Overperformed Prediction
194,Giannis Antetokounmpo,2013,15,58.93,5.100882,53.829118,Overperformed Prediction
74,Kawhi Leonard,2011,15,51.99,4.763244,47.226756,Overperformed Prediction
89,Jimmy Butler,2011,30,49.34,5.083937,44.256063,Overperformed Prediction
125,Damian Lillard,2012,6,48.29,5.448058,42.841942,Overperformed Prediction
482,Luka Dončić,2018,3,44.66,5.619060,39.040940,Overperformed Prediction
120,Anthony Davis,2012,1,51.06,16.539935,34.520065,Overperformed Prediction
206,Rudy Gobert,2013,27,36.59,3.749340,32.840660,Overperformed Prediction
9,Paul George,2010,10,39.20,8.292568,30.907432,Overperformed Prediction
490,Shai Gilgeous-Alexander,2018,11,38.61,7.961157,30.648843,Overperformed Prediction


The top names make sense: Nikola Jokić, Giannis, Kawhi, Jimmy Butler, Damian Lillard, Luka, Anthony Davis, Shai. Those are all players who became way better NBA players than a model would reasonably expect from draft position / pre-draft data.

Note the model is conservative, so it predicts closer to average outcomes. That is why stars have low predicted values compared to their actual career impact.
